In [1]:
import setup

setup.init_django()

In [2]:
from rag import (
    db as rag_db, 
    engines as rag_engines,
    settings as rag_settings, 
    updaters as rag_updaters,
)

In [3]:
from typing import Optional, Union
from sqlalchemy import create_engine, text

In [4]:
rag_settings.init()
rag_db.init_vector_db()
rag_updaters.update_llama_index_documents(use_saved_embeddings=True)

In [5]:
vector_index = rag_engines.get_semantic_query_index()
semantic_query_retriever = rag_engines.get_semantic_query_retriever_engine()
sql_query_engine = rag_engines.get_sql_query_engine()

In [6]:
print(rag_settings.VECTOR_DB_NAME, rag_settings.VECTOR_DB_TABLE_NAME)

vector_db blogpost


In [7]:
from llama_index.core.tools import QueryEngineTool

vector_tool = QueryEngineTool.from_defaults(
    query_engine=semantic_query_retriever,
    description=(
        f"Useful for answering semantic questions about different blog posts"
    ),
)

In [8]:
sql_tool = QueryEngineTool.from_defaults(
    query_engine=sql_query_engine,
    description=(
        "Useful for translating a natural language query into a SQL query over"
        " a table containing: blog posts and page views each blog post"
    ),
)

In [12]:
from typing import Any, Optional, Union


from llama_index.core.query_engine import SQLAutoVectorQueryEngine
from llama_index.core.query_engine.sql_vector_query_engine import *
from llama_index.core.service_context import ServiceContext


class MySQLAutoVectorQueryEngine(SQLAutoVectorQueryEngine):
    def __init__(
        self,
        sql_query_tool: QueryEngineTool,
        vector_query_tool: QueryEngineTool,
        selector: Optional[Union[LLMSingleSelector, PydanticSingleSelector]] = None,
        llm: Optional[LLM] = None,
        service_context: Optional[ServiceContext] = None,
        sql_vector_synthesis_prompt: Optional[BasePromptTemplate] = None,
        sql_augment_query_transform: Optional[SQLAugmentQueryTransform] = None,
        use_sql_vector_synthesis: bool = True,
        callback_manager: Optional[CallbackManager] = None,
        verbose: bool = True,
    ) -> None:
        """Initialize params."""
        # validate that the query engines are of the right type
        if not isinstance(
            sql_query_tool.query_engine,
            (BaseSQLTableQueryEngine, NLSQLTableQueryEngine),
        ):
            raise ValueError(
                "sql_query_tool.query_engine must be an instance of "
                "BaseSQLTableQueryEngine or NLSQLTableQueryEngine"
            )
        if not isinstance(vector_query_tool.query_engine, RetrieverQueryEngine):
            raise ValueError(
                "vector_query_tool.query_engine must be an instance of "
                "RetrieverQueryEngine"
            )
        # if not isinstance(
        #     vector_query_tool.query_engine.retriever, VectorIndexAutoRetriever
        # ):
        #     raise ValueError(
        #         "vector_query_tool.query_engine.retriever must be an instance "
        #         "of VectorIndexAutoRetriever"
        #     )

        sql_vector_synthesis_prompt = (
            sql_vector_synthesis_prompt or DEFAULT_SQL_VECTOR_SYNTHESIS_PROMPT
        )
        SQLJoinQueryEngine.__init__(
            self,
            sql_query_tool,
            vector_query_tool,
            selector=selector,
            llm=llm,
            
            sql_join_synthesis_prompt=sql_vector_synthesis_prompt,
            sql_augment_query_transform=sql_augment_query_transform,
            use_sql_join_synthesis=use_sql_vector_synthesis,
            callback_manager=callback_manager,
            verbose=verbose,
        )

In [13]:
# from llama_index.core.query_engine import SQLAutoVectorQueryEngine

query_engine = MySQLAutoVectorQueryEngine(
    sql_tool, 
    vector_tool,
)

In [14]:
response = query_engine.query(
    "What kind of org is discussed?"
)

Querying other query engine: The question 'What kind of org is discussed?' is a semantic question about the content of blog posts, which aligns with choice (2) that is useful for answering semantic questions about different blog posts.


In [15]:
response.response

'The discussion contrasts two types of entities: organizations and organisms. An organization is described as structured and controlled, with systems and charts that require approval for changes. In contrast, an organism is characterized by constant change, adaptation, and resilience, similar to a living culture that thrives by understanding the system it inhabits.'

In [16]:
response = query_engine.query(
    "Are are the top 5 most viewed blog posts? What keywords do their content have?"
)

Querying SQL database: The question requires translating a natural language query into a SQL query to determine the top 5 most viewed blog posts, which aligns with choice 1's focus on SQL queries over a table containing blog posts and page views.


OperationalError: (psycopg2.OperationalError) SSL SYSCALL error: EOF detected

[SQL: SELECT pg_catalog.pg_class.relname, pg_catalog.pg_description.description 
FROM pg_catalog.pg_class LEFT OUTER JOIN pg_catalog.pg_description ON pg_catalog.pg_class.oid = pg_catalog.pg_description.objoid AND pg_catalog.pg_description.objsubid = %(objsubid_1)s AND pg_catalog.pg_description.classoid = CAST(%(param_1)s AS REGCLASS) JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s, %(param_6)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s AND pg_catalog.pg_class.relname IN (%(filter_names_1)s)]
[parameters: {'objsubid_1': 0, 'param_1': 'pg_catalog.pg_class', 'param_2': 'r', 'param_3': 'p', 'param_4': 'f', 'param_5': 'v', 'param_6': 'm', 'nspname_1': 'pg_catalog', 'filter_names_1': 'blog_blogpost'

In [ ]:
from IPython.display import Markdown, display

display(Markdown(response.response))

In [ ]:
response = query_engine.query(
    "What are the top 5 least viewed blog posts from today?"
)
print(response.response)

In [ ]:
display(Markdown(response.response))